In [ ]:
import polars as pl
from abc import ABC, abstractmethod
from typing import Iterable, Dict, List
from dataclasses import dataclass

### Define labels first

In [2]:
class Label(dict):
    def matches(self, **kwargs):
        for k, v in kwargs.items():
            if k not in self.keys():
                raise ValueError(f"Trying to match on unknown namespace {k}")
            elif self[k] != v:
                return False
        return True
    
    def drop(self, *keys):
        new_mapping = {k:v for k, v in self.items() if k not in keys}
        return Label(**new_mapping)

### Define the interface for the Model Class

In [3]:
class Model(ABC):

    def fit(self, df: pl.DataFrame, *args, **kwargs) -> None:
        raise NotImplementedError

    def predict(self, df: pl.DataFrame, *args, **kwargs) -> pl.Series:
        raise NotImplementedError
    
    @property
    def labels(self) -> List[Label]:
        return [Label({'leaf': self.name})]
    
###################################
    def model_constructor(self):
        raise NotImplementedError
    
    def train_mask_for_label(self, label, *args, **kwargs):
        return True
    
    def test_mask_for_label(self, label, *args, **kwargs):
        return True
    
    def _applied_train_mask_for_label(self, label):
        raise NotImplementedError
    
    def _applied_test_mask_for_label(self, label):
        raise NotImplementedError

    @classmethod
    def __subclasshook__(cls, C):

        has_attr = lambda method: any(method in B.__dict__ for B in C.__mro__)
        
        required = [
            'fit',
            'predict',
            'test_mask_for_label',
            'train_mask_for_label',
            'model_constructor',
            'labels',
            'name',
            '_applied_train_mask_for_label',
            '_applied_test_mask_for_label',
        ]
        
        if cls is Model:
            if all([has_attr(attr) for attr in required]):
                return True

        return NotImplemented

### Define the generic nodetype

In [4]:
class Node:
   
    def __init__(self, name=None):
        self.name = name or self.__class__.__name__
        
    def __repr__(self):
        return self.name
    
    def fit(self, df: pl.DataFrame):
        raise NotImplementedError
    
    def predict(self, df: pl.DataFrame):
        raise NotImplementedError
    
    @property
    def labels(self) -> List[Label]:
        """Returns a list of all labels that are used by the node"""
        raise NotImplementedError
    
    def train_mask_for_label(self, label):
        raise NotImplementedError
    
    def test_mask_for_label(self, label):
        raise NotImplementedError
    
    def get_train_df_for_label(self, df: pl.DataFrame, label):
        return df.filter(self.train_mask_for_label(label))
    
    def get_test_df_for_label(self, df: pl.DataFrame, label):
        return df.filter(self.test_mask_for_label(label))
        

### Define the simplest operation - the lift

In [17]:
class Lift(Node):

    def __init__(self,
                 child: Model,
                 atomics: Iterable[pl.DataType],
                 name: str = None,
                 ):
        atomics = set(atomics)
        name = name or f"Lift: {atomics}"
        super().__init__(name=name)
        self.child = child
        self.models = None
        self._atomic_labels = atomics

    def train_mask_for_label(self, label) -> pl.Expr:
        raise NotImplementedError

    def test_mask_for_label(self, label) -> pl.Expr:
        raise NotImplementedError
    
    def _applied_train_mask_for_label(self, label) -> pl.Expr:
        # This combines the mask from this node with the mask
        # of the children
        lift_mask = self.train_mask_for_label(label[self.name])
        child_mask = self.child.train_mask_for_label(label.drop(self.name))
        
        return lift_mask & child_mask
        
    @property
    def labels(self):
        # Labels of a lift are the cartesian product of the labels
        # we are lifting to and the labels of the original model
        labels = []
        for child_label in self.child.labels:
            for atomic in self._atomic_labels:
                labels.append(
                    Label({**child_label, self.name: atomic})
                )

        return labels       
    
    def get_train_df_for_label(self, df: pl.DataFrame, label: Label) -> pl.DataFrame:
        return df.filter(self._applied_train_mask_for_label(label))
    
    def fit(self, df: pl.DataFrame):
        for label in self.labels:
            label_train_df = self.get_train_df_for_label(df, label)
            model = self.child.model_constructor()
            model.fit(label_train_df)
            self.models[label] = model

### Define ensembling

In [6]:
class Ensemble(Node):
    pass

### Testing

In [7]:
class PassThroughModel(Model):
    
    def __init__(self, value, name):
        self.model_constructor = lambda: None
        self.value = value
        self.name=name
 
    def fit(self):
        pass
    
    def predict(self):
        return self.value

In [8]:
class BasicLift(Lift):
     
    def train_mask_for_label(self, label) -> pl.Expr:
        return pl.col('column') == pl.lit(label)

In [9]:
model = PassThroughModel(1, 'passthrough')

In [13]:
BasicLift(model, atomics=['a', 'b', 'c']).labels

[{'leaf': 'passthrough', "Lift: {'b', 'c', 'a'}": 'b'},
 {'leaf': 'passthrough', "Lift: {'b', 'c', 'a'}": 'c'},
 {'leaf': 'passthrough', "Lift: {'b', 'c', 'a'}": 'a'}]

In [15]:
df = pl.DataFrame({'column': ['a', 'b', 'c']})

BasicLift(model, atomics=['a', 'b', 'c']).fit(df)

AttributeError: 'NoneType' object has no attribute 'fit'